# AsyncIO

Pure Python this time — no LLM calls, no `llm.py`. Every example here uses `asyncio.sleep()` standing in for a slow network call, deliberately, so nothing in this notebook depends on a provider or an API key. Later chapters apply this directly to concurrent LLM calls; this one just builds the mechanics those chapters will assume you already have.

**By the end you'll have:**
- Understood why the event loop exists in the first place
- Hit the single most common async mistake, on purpose
- Measured a real 3x speedup from running things concurrently instead of sequentially
- Seen a concrete, measurable reason `TaskGroup` is safer than `gather`
- Throttled concurrency with a `Semaphore`
- Timed out and cleanly cancelled a task that ran too long

## The event loop

Most of the time a program spends making a network call, it isn't computing anything — it's just waiting for a response. Ordinary Python blocks on that wait: nothing else can happen until it's over. The event loop is a scheduler that uses that dead time instead of wasting it: while one piece of code is waiting on I/O, the loop switches to another piece of code that's ready to make progress, and switches back once the first one's wait is over.

`async`/`await` is the syntax for writing code the event loop can do this with. Nothing here is actually parallel — it's still one thread — but I/O-bound waiting overlaps instead of stacking up.

## async and await

`async def` defines a coroutine — but defining one and running one are different things, and mixing them up is the single most common async mistake:

That's not the string `"a done"` — it's a coroutine *object*, unstarted. Calling an `async def` function doesn't run its body at all; it just builds a coroutine that something else has to actually run. `await` is that "something else":

(Jupyter lets you `await` directly in a cell — a plain `.py` script needs `asyncio.run(main())` instead, since something has to start the event loop in the first place.)

## Running things concurrently: gather

`await`-ing three things one after another still runs them one after another — `await` doesn't make anything concurrent by itself, it just doesn't block *other* code while it waits:

`asyncio.gather()` is what actually runs them concurrently — start all three, then wait for all three:

Same three calls, same total work, 3x faster wall-clock time — because all three waits overlap instead of stacking up. This is the entire reason async exists for I/O-bound code like API calls.

## TaskGroup

`gather` has a sharp edge: if one call fails, it raises immediately — but the *other* calls don't stop, they keep running in the background, wasted, whether you wanted that or not:

`gather` raised at 0.5s like you'd expect — but `a` and `c` still printed "actually finished" 1.5 seconds *after* that, well after you'd already moved on and stopped caring about the result. `TaskGroup` (Python 3.11+) doesn't have this leak — a failure cancels every sibling task immediately:

Same failure, same timing — but `a` and `c` never print "actually finished," because `TaskGroup` cancelled them the moment `b` raised. That's a real difference in behavior, not just style: no wasted API calls, no dangling work. `except*` is new syntax (an *exception group* handler) because a `TaskGroup` can have several tasks fail at once — `eg.exceptions` is a tuple of everything that went wrong, not just one.

## Semaphore

Concurrency isn't free — a real API has rate limits, and firing off hundreds of calls with `gather`/`TaskGroup` unthrottled will get you 429s. A `Semaphore(n)` caps how many tasks can be *in progress* at once, without capping how many you can queue up:

Four 1-second tasks, limit of 2 at a time: `a`/`b` start immediately, `c`/`d` wait until one of the first two releases the semaphore — 2 seconds total, not 1 (fully concurrent) or 4 (sequential). Same tasks, same `TaskGroup`, just throttled.

## Cancellation

Sometimes a call just takes too long and you need to give up on it. `asyncio.wait_for()` cancels the underlying coroutine when the timeout hits — the coroutine actually gets a chance to notice and clean up:

`slow_fetch` was 5 seconds from finishing, but the caller only waited 1: `wait_for` cancelled it, `CancelledError` propagated *into* the coroutine so it could react, then `TimeoutError` surfaced at the call site. Re-raising `CancelledError` after cleanup (rather than swallowing it) is the correct pattern — it's what lets cancellation actually complete instead of getting stuck.

## What you built

✓ Understood why the event loop exists: overlapping I/O waits instead of stacking them

✓ Hit the "forgot to await" mistake directly — a coroutine object, not a result

✓ Measured a real 3x speedup from `gather` over sequential `await`

✓ Proved `TaskGroup` cancels sibling tasks on failure where `gather` lets them run to waste

✓ Throttled concurrency to 2-at-a-time with a `Semaphore`

✓ Timed out and cleanly cancelled a task with `wait_for`

No `llm.py` changes here on purpose — this chapter is Python mechanics, not an LLM feature. The concurrency patterns above (especially `TaskGroup` + `Semaphore`) are exactly what later chapters reach for once there's a real reason to make several LLM calls at once.

**Next:** `09_build_an_agent.ipynb` — combining every notebook in this chapter into an agent loop with real memory.